<a href="https://colab.research.google.com/github/vhousos/EKPA/blob/main/Chousos_Final_Exam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Στο παρόν notebook υλοποιείται το πρώτο επίπεδο ενός ευφυούς συστήματος κυβερνοασφάλειας
για περιβάλλον κυβερνητικού Security Operations Center (SOC).

Στόχος του πρώτου επιπέδου είναι:
- η προετοιμασία (preprocessing) των δεδομένων δικτυακών ροών,
- και η εκπαίδευση ενός μοντέλου μηχανικής μάθησης για την ανίχνευση
  μη εξουσιοδοτημένης Tor-based επικοινωνίας (Tor vs Non-Tor).

Η ανάλυση βασίζεται αποκλειστικά σε network flow metadata, χωρίς επιθεώρηση περιεχομένου, σύμφωνα με ρεαλιστικές πρακτικές SOC.

**Εισαγωγή βιβλιοθηκών**

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


**Φόρτωση CSV**

In [3]:
# Φόρτωση του dataset απευθείας από το GitHub repository
url = "https://raw.githubusercontent.com/kdemertzis/EKPA/main/Data/DarkNet.csv"

df = pd.read_csv(url)

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (68580, 83)


/tmp/ipython-input-3992536184.py:4: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url)


,Src_IP,Src_Port,Dst_IP,Dst_Port,Protocol,Flow_Duration,Total_Fwd_Packet,Total_Bwd_packets,Total_Length_of_Fwd_Packet,Total_Length_of_Bwd_Packet,...,Active_Mean,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Label-1,Label-2
0,10.152.152.11,57158,216.58.220.99,443,6,229,1,1,0,0,...,0,0,0,0,0,0.000,0,0,Non-Tor,AUDIO-STREAMING
1,10.152.152.11,57159,216.58.220.99,443,6,407,1,1,0,0,...,0,0,0,0,0,0.000,0,0,Non-Tor,AUDIO-STREAMING
2,10.152.152.11,57160,216.58.220.99,443,6,431,1,1,0,0,...,0,0,0,0,0,0.000,0,0,Non-Tor,AUDIO-STREAMING
3,10.152.152.11,49134,74.125.136.120,443,6,359,1,1,0,0,...,0,0,0,0,0,0.000,0,0,Non-Tor,AUDIO-STREAMING
4,10.152.152.11,34697,173.194.65.127,19305,6,10778451,591,400,64530,6659,...,0,0,0,0,1437760000000000,3117718.131,1437760000000000,1437760000000000,Non-Tor,AUDIO-STREAMING


**Αρχικός έλεγχος δεδομένων**
πραγματοποιείται ένας βασικός έλεγχος:
- δομής του dataset,
- τύπων δεδομένων,
- και κατανομής των labels.

για να επιβεβαιώσουμε ότι τα δεδομένα είναι κατάλληλα για χρήση σε pipeline μηχανικής μάθησης.

**Sanity checks**

In [4]:
df.info()
df["Label-1"].value_counts(dropna=False)
df["Label-2"].value_counts(dropna=False)



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68580 entries, 0 to 68579
Data columns (total 83 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Src_IP                      68580 non-null  object 
 1   Src_Port                    68580 non-null  int64  
 2   Dst_IP                      68580 non-null  object 
 3   Dst_Port                    68580 non-null  int64  
 4   Protocol                    68580 non-null  int64  
 5   Flow_Duration               68580 non-null  int64  
 6   Total_Fwd_Packet            68580 non-null  int64  
 7   Total_Bwd_packets           68580 non-null  int64  
 8   Total_Length_of_Fwd_Packet  68580 non-null  int64  
 9   Total_Length_of_Bwd_Packet  68580 non-null  int64  
 10  Fwd_Packet_Length_Max       68580 non-null  int64  
 11  Fwd_Packet_Length_Min       68580 non-null  int64  
 12  Fwd_Packet_Length_Mean      68580 non-null  float64
 13  Fwd_Packet_Length_Std       685

,count
Label-2,
P2P,13711
Chat,11478
File-Transfer,11098
Video-Streaming,9486
Email,6145
Audio-Streaming,6055
Browsing,5192
VOIP,3566
AUDIO-STREAMING,1484


**Minimal preprocessing**
Η προεπεξεργασία που εφαρμόζεται είναι σκόπιμα περιορισμένη

Συγκεκριμένα:
- Αφαιρούνται διευθύνσεις IP, καθώς αποτελούν αναγνωριστικά και μπορούν να οδηγήσουν
  σε διαρροή πληροφορίας (data leakage) και υπερπροσαρμογή (overfitting).
- Αντικαθίστανται άπειρες τιμές (infinite) που προκύπτουν από ροές μηδενικής διάρκειας.
- Δεν αφαιρούνται ακραίες τιμές (outliers), καθώς  συχνά αποτελούν ένδειξη ανώμαλης ή ύποπτης συμπεριφοράς.

In [5]:
# Αφαίρεση IP διευθύνσεων (identifiers – όχι κατάλληλα για ML)
drop_cols = [c for c in ["Src_IP", "Dst_IP"] if c in df.columns]
df = df.drop(columns=drop_cols)

# Αντικατάσταση infinite τιμών με NaN
df = df.replace([np.inf, -np.inf], np.nan)


**Ορισμός features & target για Level 1**
Για το πρώτο επίπεδο του συστήματος:
- Ως στόχος (target variable) χρησιμοποιείται η ετικέτα Label-1, η οποία υποδεικνύει εάν μία δικτυακή ροή σχετίζεται με Tor ή όχι.
- Η ετικέτα Label-2 (τύπος δραστηριότητας) δεν χρησιμοποιείται στο Level 1, ώστε να αποφευχθεί διαρροή πληροφορίας.

In [6]:
y = df["Label-1"].astype(str)

X = df.drop(columns=["Label-1", "Label-2"])
print("Features shape:", X.shape)
print("Target shape:", y.shape)


Features shape: (68580, 79)
Target shape: (68580,)


**Διαχωρισμός train / test**
Τα δεδομένα χωρίζονται σε:
- 80% για εκπαίδευση,
- 20% για έλεγχο.

Ο διαχωρισμός γίνεται με stratification, ώστε να διατηρηθεί η αναλογία Tor / Non-Tor

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train set:", X_train.shape)
print("Test set:", X_test.shape)


Train set: (54864, 79)
Test set: (13716, 79)


**Pipeline προεπεξεργασίας χαρακτηριστικών**
Για την εκπαίδευση του μοντέλου:
- Τα αριθμητικά χαρακτηριστικά συμπληρώνονται με median imputation,καθώς ι κατανομές είναι έντονα ασύμμετρες.
- Τα κατηγορικά χαρακτηριστικά (π.χ. Protocol) κωδικοποιούνται με one-hot encoding.

Όλη η προεπεξεργασία ενσωματώνεται σε ενιαίο pipeline, ώστε να διασφαλίζεται:
- αναπαραγωγιμότητα,
- καθαρή ροή δεδομένων,
- και σωστή εφαρμογή των ίδιων μετασχηματισμών
  σε δεδομένα εκπαίδευσης και ελέγχου.

In [8]:
categorical_cols = [c for c in X.columns if X[c].dtype == "object"]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)


**Εκπαίδευση Level 1 μοντέλου**
Για το πρώτο επίπεδο χρησιμοποιείται Random Forest Classifier, γιατί αποδίδει καλά σε tabular network flow δεδομένα, δεν απαιτεί scaling, και είναι ανθεκτικός σε θόρυβο και ανισορροπία κλάσεων.

Το μοντέλο εκπαιδεύεται για να διακρίνει Tor από Non-Tor traffic,
λειτουργώντας ως το αρχικό φίλτρο του SOC.

In [9]:
from sklearn.preprocessing import FunctionTransformer

# Identify categorical/numeric columns
categorical_cols = [c for c in X.columns if X[c].dtype == "object"]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

# Numeric pipeline
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical pipeline: impute -> cast to string -> one-hot
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("to_str", FunctionTransformer(lambda a: a.astype(str), feature_names_out="one-to-one")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop"
)


## Αξιολόγηση απόδοσης μοντέλου Level 1

Η απόδοση του μοντέλου αξιολογείται στο σύνολο ελέγχου, με στόχο να εκτιμηθεί η αποτελεσματικότητα της ανίχνευσης Tor traffic σε συνθήκες προσομοίωσης SOC.


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Identify categorical/numeric columns from X (same X you use for split)
categorical_cols = [c for c in X.columns if X[c].dtype == "object"]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("to_str", FunctionTransformer(lambda a: a.astype(str), feature_names_out="one-to-one")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

# Build a fresh model pipeline and refit (required after rebuilding preprocessor)
model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample"
    ))
])

model.fit(X_train, y_train)

# Now evaluation will work
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

     Non-Tor       1.00      1.00      1.00      6186
      NonVPN       0.95      0.96      0.95      4548
         Tor       0.98      0.88      0.93       279
         VPN       0.94      0.92      0.93      2703

    accuracy                           0.97     13716
   macro avg       0.97      0.94      0.95     13716
weighted avg       0.97      0.97      0.97     13716

Confusion Matrix:
 [[6169   13    0    4]
 [  10 4373    5  160]
 [   4   30  245    0]
 [   8  204    0 2491]]


**Υπολογισμός ROC-AUC**
Η μετρική ROC-AUC χρησιμοποιείται για να εκτιμηθεί η   ικανότητα του μοντέλου στην αναγνώριση Tor traffic ως θετική κλάση.

In [11]:
# Πιθανότητες πρόβλεψης
proba = model.predict_proba(X_test)

# Οι κλάσεις με τη σειρά που τις βλέπει ο classifier
classes = list(model.named_steps["classifier"].classes_)
print("Classes:", classes)

# Βρίσκουμε ποια κλάση αντιστοιχεί στο "Tor"
if "Tor" in classes:
    tor_index = classes.index("Tor")
    tor_scores = proba[:, tor_index]

    # Binary ground truth: Tor=1, αλλιώς 0
    y_true_bin = (y_test.astype(str) == "Tor").astype(int)

    roc_auc = roc_auc_score(y_true_bin, tor_scores)
    print("ROC-AUC (Tor ως θετική κλάση):", roc_auc)
else:
    print("Δεν βρέθηκε κλάση 'Tor' στο y. Ελέγξτε τα labels του Label-1.")

Classes: ['Non-Tor', 'NonVPN', 'Tor', 'VPN']
ROC-AUC (Tor ως θετική κλάση): 0.9971186391398276


**LEVEL 2 — Tor-only Enrichment για SOC**
Το Level 1 λειτουργεί ως αρχικό φίλτρο και εντοπίζει ροές που σχετίζονται με Tor.

Στο Level 2 εφαρμόζεται ανάλυση μόνο σε ροές που έχουν χαρακτηριστεί ως Tor, με δύο στόχους:

(2B) Risk / Anomaly Scoring
- Απόδοση risk score σε Tor flows ώστε το SOC να ιεραρχεί τα περιστατικά
  και να αποφεύγει alert fatigue.

(2A) Traffic Profiling, συμπληρωματικα
- Εκτίμηση του τύπου δραστηριότητας (Label-2) μέσα στο Tor traffic,
  ώστε ο αναλυτής να έχει γρηγορότερο context για διαλογή και ιεράρχηση περιστατικών (triage).

Για ρεαλιστική SOC προσέγγιση, το Level 2 μπορεί να δουλέψει με δύο τρόπους:
1) **Operational mode**: Χρησιμοποιεί τα Tor flows όπως τα προβλέπει το Level 1 (πιο ρεαλιστικό).
2) **Training/Evaluation mode**: Χρησιμοποιεί το ground truth (Label-1 == "Tor") για να εκπαιδεύσει/ελέγξει σταθερά.

Παρακάτω υλοποιούμε και τα δύο:
- `tor_pred_df` (Tor flows που προβλέπει το Level 1)
- `tor_true_df` (Tor flows από το ground truth)

In [12]:
# X: features (χωρίς Label-1/Label-2), όπως στο Level 1
# df: το πλήρες dataframe που περιέχει Label-1 και Label-2
# model: trained Level 1 pipeline

# Προβλέψεις Level 1 σε ΟΛΑ τα flows (operational view)
pred_all = model.predict(X)

tor_pred_mask = (pred_all.astype(str) == "Tor")
tor_true_mask = (df["Label-1"].astype(str) == "Tor")

tor_pred_df = df.loc[tor_pred_mask].copy()
tor_true_df = df.loc[tor_true_mask].copy()

print("Tor flows (predicted by Level 1):", tor_pred_df.shape[0])
print("Tor flows (ground truth Label-1):", tor_true_df.shape[0])

# Features για Level 2: αφαιρούμε labels (και IPs έχουν ήδη αφαιρεθεί)
tor_pred_X = tor_pred_df.drop(columns=[c for c in ["Label-1", "Label-2"] if c in tor_pred_df.columns])
tor_true_X = tor_true_df.drop(columns=[c for c in ["Label-1", "Label-2"] if c in tor_true_df.columns])


Tor flows (predicted by Level 1): 1367
Tor flows (ground truth Label-1): 1392


**(2B) Risk / Anomaly Scoring**

Στόχος: Για κάθε Tor flow να υπολογίζουμε έναν δείκτη "απόκλισης" από το συνηθισμένο Tor behavior.

H λογική λέει:
- Δεν χαρακτηρίζουμε το Tor ως κακόβουλο από μόνο του.
- Χρησιμοποιούμε anomaly/risk score για να δούμε ποια Tor flows είναι "πιο ασυνήθιστα" και πρέπει να διερευνηθούν κατά προτεραιότητα.

Προσέγγιση:
- Εκπαίδευση unsupervised μοντέλου σε Tor flows (ground truth Tor),  και εφαρμογή scoring στα Tor flows που εντοπίζει operationally το Level 1.

In [13]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Categorical/numeric split (πάνω στα Level2 features)
categorical_cols_l2 = [c for c in tor_true_X.columns if tor_true_X[c].dtype == "object"]
numeric_cols_l2 = [c for c in tor_true_X.columns if c not in categorical_cols_l2]

numeric_transformer_l2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer_l2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("to_str", FunctionTransformer(lambda a: a.astype(str), feature_names_out="one-to-one")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_l2 = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_l2, numeric_cols_l2),
        ("cat", categorical_transformer_l2, categorical_cols_l2),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)


**Εκπαίδευση Anomaly Model (Isolation Forest)**
Χρησιμοποιούμε Isolation Forest γιατί είναι κατάλληλο για tabular δεδομένα, δουλεύει καλά σε high-dimensional φeatures, και είναι συχνή επιλογή για SOC anomaly scoring.

Το μοντέλο εκπαιδεύεται σε Tor flows (ground truth),
ώστε να "μάθει" το συνηθισμένο Tor behavior.

Στη συνέχεια εφαρμόζεται scoring στα Tor flows που εντοπίζει το Level 1,
για operational ιεράρχηση.

In [14]:
from sklearn.ensemble import IsolationForest

# Μετασχηματισμός features
X_tor_true_mat = preprocessor_l2.fit_transform(tor_true_X)
X_tor_pred_mat = preprocessor_l2.transform(tor_pred_X)

iso = IsolationForest(
    n_estimators=300,
    random_state=42,
    contamination="auto",   # δεν υποθέτουμε % ανωμαλιών
    n_jobs=-1
)

iso.fit(X_tor_true_mat)

# decision_function: μεγαλύτερο = πιο "normal"
# anomaly score: αντιστρέφουμε ώστε μεγαλύτερο = πιο "ύποπτο"
tor_pred_anom_raw = -iso.decision_function(X_tor_pred_mat)

# Κλιμάκωση σε Risk Score 0–100 για SOC-friendly output
raw = tor_pred_anom_raw
risk_0_100 = 100 * (raw - raw.min()) / (raw.max() - raw.min() + 1e-12)

tor_pred_df["Tor_Anomaly_Score_Raw"] = raw
tor_pred_df["Tor_Risk_Score_0_100"] = risk_0_100

def level2_filter(d):
    d = d.copy()
    if "Flow_Duration" in d.columns:
        d = d[d["Flow_Duration"].fillna(0) > 0]
    if "Protocol" in d.columns:
        d = d[d["Protocol"].fillna(0) != 0]
    return d

tor_pred_df = level2_filter(tor_pred_df)
tor_true_df = level2_filter(tor_true_df)

print("Tor flows (predicted, after L2 filter):", tor_pred_df.shape[0])
print("Tor flows (true, after L2 filter):", tor_true_df.shape[0])

tor_pred_X = tor_pred_df.drop(columns=[c for c in ["Label-1", "Label-2"] if c in tor_pred_df.columns])
tor_true_X = tor_true_df.drop(columns=[c for c in ["Label-1", "Label-2"] if c in tor_true_df.columns])

# Μετασχηματισμός features
X_tor_true_mat = preprocessor_l2.fit_transform(tor_true_X)
X_tor_pred_mat = preprocessor_l2.transform(tor_pred_X)

iso = IsolationForest(
    n_estimators=300,
    random_state=42,
    contamination="auto",
    n_jobs=-1
)

iso.fit(X_tor_true_mat)

# decision_function: μεγαλύτερο = πιο "normal"
# anomaly score: αντιστρέφουμε ώστε μεγαλύτερο = πιο "ύποπτο"
tor_pred_anom_raw = -iso.decision_function(X_tor_pred_mat)

# Κλιμάκωση σε Risk Score 0–100
raw = tor_pred_anom_raw
risk_0_100 = 100 * (raw - raw.min()) / (raw.max() - raw.min() + 1e-12)

tor_pred_df["Tor_Anomaly_Score_Raw"] = raw
tor_pred_df["Tor_Risk_Score_0_100"] = risk_0_100


tor_pred_df[["Tor_Risk_Score_0_100"]].describe()
tor_pred_df.sort_values("Tor_Risk_Score_0_100", ascending=False).head(25)



Tor flows (predicted, after L2 filter): 1329
Tor flows (true, after L2 filter): 1357


,Src_Port,Dst_Port,Protocol,Flow_Duration,Total_Fwd_Packet,Total_Bwd_packets,Total_Length_of_Fwd_Packet,Total_Length_of_Bwd_Packet,Fwd_Packet_Length_Max,Fwd_Packet_Length_Min,...,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Label-1,Label-2,Tor_Anomaly_Score_Raw,Tor_Risk_Score_0_100
57333,138,138,17,118093644,3,0,573,0,191,191,...,0,0,1427990000000000,0.00,1427990000000000,1427990000000000,Tor,Video-Streaming,-0.103387,100.000000
57353,138,138,17,118119863,4,0,764,0,191,191,...,0,0,1427990000000000,77920609.68,1427990000000000,1427990000000000,Tor,Video-Streaming,-0.103958,99.387862
57369,138,138,17,86256086,4,0,764,0,191,191,...,0,0,1427990000000000,55054069.68,1427990000000000,1427990000000000,Tor,Video-Streaming,-0.117000,85.391140
67357,47131,80,6,9707315,2,0,0,0,0,0,...,0,0,0,0.00,0,0,Tor,VOIP,-0.125125,76.671867
67358,39212,80,6,9899348,2,0,0,0,0,0,...,0,0,0,0.00,0,0,Tor,VOIP,-0.125703,76.051540
32817,36922,443,6,119952629,112274,80558,141400896,4471215,1460,0,...,0,0,1437150000000000,35117551.65,1437150000000000,1437150000000000,Tor,File-Transfer,-0.125996,75.737442
32816,36922,443,6,119998538,99421,71413,125555103,3994063,1460,0,...,0,0,1437150000000000,34669680.89,1437150000000000,1437150000000000,Tor,File-Transfer,-0.127912,73.680968
32820,36922,443,6,119961884,113960,81630,143491751,4534472,1460,0,...,0,0,1437150000000000,34446049.98,1437150000000000,1437150000000000,Tor,File-Transfer,-0.127998,73.588657
32849,36922,443,6,119578967,104789,76695,130842675,7308047,1460,0,...,0,0,1437150000000000,32253399.69,1437150000000000,1437150000000000,Tor,File-Transfer,-0.130531,70.870352
57332,9001,34160,6,119952316,38057,23218,62537552,2052541,29436,0,...,0,0,1427990000000000,33908382.17,1427990000000000,1427990000000000,Tor,Video-Streaming,-0.130589,70.808120


**Top risky Tor flows**
Παρακάτω εμφανίζουμε τα πιο "ύποπτα" Tor flows (υψηλότερο risk score),
ώστε να προσομοιώσουμε ροή triage σε SOC.

In [16]:
top_n = 25
cols_to_show = [c for c in ["Protocol", "Dst_Port", "Flow_Duration", "Flow_Bytes/s", "Flow_Packets/s"] if c in tor_pred_df.columns]
cols_to_show = cols_to_show + ["Tor_Risk_Score_0_100"]

tor_pred_df.sort_values("Tor_Risk_Score_0_100", ascending=False).head(top_n)[cols_to_show]


,Protocol,Dst_Port,Flow_Duration,Flow_Bytes/s,Flow_Packets/s,Tor_Risk_Score_0_100
57333,17,138,118093644,4.852082,0.025404,100.000000
57353,17,138,118119863,6.468006,0.033864,99.387862
57369,17,138,86256086,8.857346,0.046374,85.391140
67357,6,80,9707315,0.0,0.206030,76.671867
67358,6,80,9899348,0.0,0.202034,76.051540
32817,6,443,119952629,1216080.983,1607.567934,75.737442
32816,6,443,119998538,1079589.536,1423.634011,73.680968
32820,6,443,119961884,1233943.8,1630.434547,73.588657
32849,6,443,119578967,1155309.545,1517.691652,70.870352
57332,6,34160,119952316,538464.7429,510.827986,70.808120


**Traffic profiling μέσα στο Tor**
(2A) Traffic Profiling μέσα στο Tor (Label-2)
τόχος: Για Tor flows, να εκτιμήσουμε τον τύπο δραστηριότητας (Label-2),
π.χ. AUDIO-STREAMING, CHAT κ.λπ., ώστε το SOC να έχει context.

Εδώ κάνουμε supervised classification μέσα στο Tor subset:
- Εκπαίδευση σε (Tor ground truth flows) με target = Label-2
- Χρήση παρόμοιου pipeline με το Level 1 (Random Forest baseline)

Αυτό είναι συμπληρωματικό: ο κύριος επιχειρησιακός στόχος του Level 2 παραμένει το risk scoring.

In [17]:
# Χρησιμοποιούμε ground truth Tor για training του profiling
tor_prof_df = tor_true_df.copy()

y_prof = tor_prof_df["Label-2"].astype(str)
X_prof = tor_prof_df.drop(columns=[c for c in ["Label-1", "Label-2"] if c in tor_prof_df.columns])

print("Profiling dataset:", X_prof.shape, y_prof.shape)
y_prof.value_counts().head(10)


Profiling dataset: (1357, 79) (1357,)


,count
Label-2,
VOIP,297
Browsing,251
Audio-Streaming,222
P2P,220
Video-Streaming,189
File-Transfer,107
Chat,58
Email,13


**Train/test split + pipeline + train**

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X_prof_train, X_prof_test, y_prof_train, y_prof_test = train_test_split(
    X_prof, y_prof,
    test_size=0.2,
    random_state=42,
    stratify=y_prof
)

# Reuse preprocessor_l2 logic but fit on profiling features
categorical_cols_prof = [c for c in X_prof.columns if X_prof[c].dtype == "object"]
numeric_cols_prof = [c for c in X_prof.columns if c not in categorical_cols_prof]

numeric_transformer_prof = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer_prof = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("to_str", FunctionTransformer(lambda a: a.astype(str), feature_names_out="one-to-one")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_prof = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_prof, numeric_cols_prof),
        ("cat", categorical_transformer_prof, categorical_cols_prof),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

prof_model = Pipeline(steps=[
    ("preprocess", preprocessor_prof),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample"
    ))
])

prof_model.fit(X_prof_train, y_prof_train)

y_prof_pred = prof_model.predict(X_prof_test)
print(classification_report(y_prof_test, y_prof_pred))


                 precision    recall  f1-score   support

Audio-Streaming       0.88      0.98      0.92        44
       Browsing       0.84      0.82      0.83        50
           Chat       1.00      0.75      0.86        12
          Email       1.00      0.67      0.80         3
  File-Transfer       1.00      0.81      0.89        21
            P2P       0.98      0.95      0.97        44
           VOIP       0.93      0.88      0.91        60
Video-Streaming       0.76      0.92      0.83        38

       accuracy                           0.89       272
      macro avg       0.92      0.85      0.88       272
   weighted avg       0.90      0.89      0.89       272



**Apply profiling to operational Tor flows**
Αυτό το βήμα προσομοιώνει το SOC enrichment:
- Το Level 1 εντοπίζει Tor flows
- Το Level 2 προσθέτει - risk score & - εκτίμηση traffic type (Label-2 prediction)

Στόχος είναι η παραγωγή μίας "πλουσιότερης" εγγραφής συμβάντος για SIEM/SOC triage.

In [19]:
tor_pred_df["Pred_Traffic_Type"] = prof_model.predict(tor_pred_X)

tor_pred_df[["Tor_Risk_Score_0_100", "Pred_Traffic_Type"]].head(10)


,Tor_Risk_Score_0_100,Pred_Traffic_Type
6413,8.878338,Browsing
6414,5.302187,Browsing
6417,10.344514,Browsing
6418,13.430613,Browsing
6419,7.113769,Browsing
6420,11.739310,Browsing
6421,22.914489,Browsing
6422,11.506699,Browsing
6423,8.137690,Browsing
6424,26.196360,Browsing


**Τελικό SOC output view**

Παρακάτω δημιουργούμε ένα ενδεικτικό view που θα μπορούσε να σταλεί σε SIEM:
- Tor flag (από Level 1)
- Risk score (Level 2B)
- Predicted traffic type (Level 2A)
- Βασικά flow metrics για triage

In [20]:
soc_cols = [c for c in ["Protocol", "Dst_Port", "Flow_Duration", "Flow_Bytes/s", "Flow_Packets/s"] if c in tor_pred_df.columns]
soc_cols += ["Tor_Risk_Score_0_100", "Pred_Traffic_Type"]

tor_pred_df.sort_values("Tor_Risk_Score_0_100", ascending=False).head(25)[soc_cols]


,Protocol,Dst_Port,Flow_Duration,Flow_Bytes/s,Flow_Packets/s,Tor_Risk_Score_0_100,Pred_Traffic_Type
57333,17,138,118093644,4.852082,0.025404,100.000000,Video-Streaming
57353,17,138,118119863,6.468006,0.033864,99.387862,Video-Streaming
57369,17,138,86256086,8.857346,0.046374,85.391140,Video-Streaming
67357,6,80,9707315,0.0,0.206030,76.671867,VOIP
67358,6,80,9899348,0.0,0.202034,76.051540,VOIP
32817,6,443,119952629,1216080.983,1607.567934,75.737442,File-Transfer
32816,6,443,119998538,1079589.536,1423.634011,73.680968,File-Transfer
32820,6,443,119961884,1233943.8,1630.434547,73.588657,File-Transfer
32849,6,443,119578967,1155309.545,1517.691652,70.870352,File-Transfer
57332,6,34160,119952316,538464.7429,510.827986,70.808120,Video-Streaming
